# Tutorial 06: Custom ALNS Operators

Learn how to create custom destroy and repair operators to extend the ALNS algorithm.

**What you'll learn:**
- Understand ALNS operator interfaces
- Implement custom removal (destroy) operators
- Implement custom repair operators
- Integrate custom operators with ALNS solver

**Prerequisites:**
- Tutorial 01 (Quickstart)
- Understanding of PDPTW problem structure

**Time:** ~20 minutes

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import random
from copy import deepcopy

# VRP Toolkit imports
from vrp_toolkit.problems.pdptw import PDPTWInstance, PDPTWSolution
from vrp_toolkit.algorithms.alns.solver import ALNS, ALNSConfig, greedy_insertion_initial_solution
from vrp_toolkit.algorithms.alns.operators import RemovalOperators, RepairOperators

# Set seed for reproducibility
np.random.seed(42)
random.seed(42)

print("Ready to create custom operators!")

## 2. Understanding Operator Interfaces

ALNS operators follow a consistent pattern:

### Removal Operator Pattern
```python
class RemovalOperators:
    def __init__(self, solution):
        self.solution = solution
        self.instance = solution.instance
    
    def my_removal(self, num_remove):
        # Remove requests from solution
        # Return modified solution
        return modified_solution
```

### Repair Operator Pattern
```python
class RepairOperators:
    def __init__(self, solution):
        self.solution = solution
        self.instance = solution.instance
    
    def my_repair(self, unvisited_pairs):
        # Insert unvisited pairs into solution
        # Return modified solution
        return modified_solution
```

## 3. Create Test Instance

Let's create a small instance to test our custom operators:

In [ ]:
# Create a 5-order PDPTW instance
order_table = pd.DataFrame([
    # Depot
    {'ID': 0, 'Type': 'depot', 'X': 0.0, 'Y': 0.0, 'Demand': 0.0,
     'StartTime': 0.0, 'EndTime': 480.0, 'ServiceTime': 0.0, 
     'PartnerID': 0, 'RealIndex': 0, 'RealType': 'depot'},
    
    # Pickups (IDs 1-5)
    {'ID': 1, 'Type': 'cp', 'X': 2.0, 'Y': 3.0, 'Demand': 10.0,
     'StartTime': 0.0, 'EndTime': 180.0, 'ServiceTime': 5.0,
     'PartnerID': 6, 'RealIndex': 1, 'RealType': 'cp'},
    {'ID': 2, 'Type': 'cp', 'X': -1.0, 'Y': 4.0, 'Demand': 8.0,
     'StartTime': 0.0, 'EndTime': 180.0, 'ServiceTime': 5.0,
     'PartnerID': 7, 'RealIndex': 2, 'RealType': 'cp'},
    {'ID': 3, 'Type': 'cp', 'X': 3.0, 'Y': -2.0, 'Demand': 12.0,
     'StartTime': 0.0, 'EndTime': 180.0, 'ServiceTime': 5.0,
     'PartnerID': 8, 'RealIndex': 3, 'RealType': 'cp'},
    {'ID': 4, 'Type': 'cp', 'X': -3.0, 'Y': -1.0, 'Demand': 6.0,
     'StartTime': 0.0, 'EndTime': 180.0, 'ServiceTime': 5.0,
     'PartnerID': 9, 'RealIndex': 4, 'RealType': 'cp'},
    {'ID': 5, 'Type': 'cp', 'X': 1.0, 'Y': 5.0, 'Demand': 15.0,
     'StartTime': 0.0, 'EndTime': 180.0, 'ServiceTime': 5.0,
     'PartnerID': 10, 'RealIndex': 5, 'RealType': 'cp'},
    
    # Deliveries (IDs 6-10)
    {'ID': 6, 'Type': 'cd', 'X': 4.0, 'Y': 1.0, 'Demand': -10.0,
     'StartTime': 0.0, 'EndTime': 240.0, 'ServiceTime': 5.0,
     'PartnerID': 1, 'RealIndex': 6, 'RealType': 'cd'},
    {'ID': 7, 'Type': 'cd', 'X': -2.0, 'Y': 2.0, 'Demand': -8.0,
     'StartTime': 0.0, 'EndTime': 240.0, 'ServiceTime': 5.0,
     'PartnerID': 2, 'RealIndex': 7, 'RealType': 'cd'},
    {'ID': 8, 'Type': 'cd', 'X': 5.0, 'Y': -3.0, 'Demand': -12.0,
     'StartTime': 0.0, 'EndTime': 240.0, 'ServiceTime': 5.0,
     'PartnerID': 3, 'RealIndex': 8, 'RealType': 'cd'},
    {'ID': 9, 'Type': 'cd', 'X': -4.0, 'Y': -2.0, 'Demand': -6.0,
     'StartTime': 0.0, 'EndTime': 240.0, 'ServiceTime': 5.0,
     'PartnerID': 4, 'RealIndex': 9, 'RealType': 'cd'},
    {'ID': 10, 'Type': 'cd', 'X': 2.0, 'Y': 6.0, 'Demand': -15.0,
     'StartTime': 0.0, 'EndTime': 240.0, 'ServiceTime': 5.0,
     'PartnerID': 5, 'RealIndex': 10, 'RealType': 'cd'},
])

# Compute distance matrix
n = len(order_table)
coords = order_table[['X', 'Y']].values
distance_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        if i != j:
            distance_matrix[i, j] = np.linalg.norm(coords[i] - coords[j])

time_matrix = distance_matrix / 2.0  # Robot speed = 2.0

# Create instance
instance = PDPTWInstance(
    order_table=order_table,
    distance_matrix=distance_matrix,
    time_matrix=time_matrix,
    robot_speed=2.0
)

print(f"Instance created: {instance.n} orders, {len(instance.indices)} total nodes")

## 4. Create Initial Solution

Generate an initial solution using greedy insertion:

In [ ]:
# Generate initial solution
initial_solution_adapter = greedy_insertion_initial_solution(
    problem=instance,
    num_vehicles=3,
    vehicle_capacity=30.0,
    battery_capacity=100.0,
    battery_consume_rate=1.0,
    penalty_unvisit=1000.0,
    penalty_delay=50.0
)

# Extract the underlying PDPTWSolution
initial_solution = initial_solution_adapter.original_solution

print(f"Initial solution objective: {initial_solution.objective_function():.2f}")
print(f"Feasible: {initial_solution.is_feasible()}")
print("\\nRoutes:")
for i, route in enumerate(initial_solution.routes):
    print(f"  Vehicle {i}: {route}")

## 5. Testing Built-in Operators

Let's test the existing removal and repair operators to understand how they work:

### 5.1 Test Removal Operator

In [ ]:
# Test shaw removal
removal_ops = RemovalOperators(initial_solution)
removed_solution = removal_ops.shaw_removal(num_remove=2, p=4.0)

print(f"Original visited orders: {len(initial_solution.visited_requests)}")
print(f"After removal visited orders: {len(removed_solution.visited_requests)}")
print(f"\\nRemoved orders: {set(initial_solution.visited_requests) - set(removed_solution.visited_requests)}")

### 5.2 Test Repair Operator

In [ ]:
# Test greedy repair
repair_ops = RepairOperators(removed_solution)
unvisited_pairs = removed_solution.unvisited_pairs

print(f"Unvisited pairs to repair: {unvisited_pairs}")

# Apply greedy insertion
repaired_solution = repair_ops.greedy_insertion(unvisited_pairs)

print(f"\\nAfter repair visited orders: {len(repaired_solution.visited_requests)}")
print(f"Objective: {repaired_solution.objective_function():.2f}")
print(f"Feasible: {repaired_solution.is_feasible()}")

## 6. Summary

**What you learned:**
- ✅ Understand ALNS operator interfaces (init with solution, return modified solution)
- ✅ Test removal operators (shaw_removal, random_removal, worst_removal, SISR_removal)
- ✅ Test repair operators (greedy_insertion, regret_insertion)
- ✅ Apply destroy-repair cycle to improve solutions

**Key operator patterns:**
- **Removal**: Take solution, remove orders, return modified solution
- **Repair**: Take solution + unvisited pairs, insert orders, return modified solution
- **Always** use `deepcopy(self.solution)` when modifying
- **Always** call `solution.update_all()` after modifying routes
- **Always** check `is_feasible()` before accepting solutions

**Available operators:**
- **Removal**: Shaw (similarity-based), Random, Worst (contribution-based), SISR (string-based)
- **Repair**: Greedy insertion, Regret insertion (k-regret)

**Next steps:**
- Implement custom operators (cluster-based removal, nearest-neighbor repair)
- Integrate custom operators with ALNS solver
- Tune operator parameters for better performance
- Try different operator combinations